In [2]:
# !pip install selenium webdriver-manager beautifulsoup4 pandas lxml

import re
import json
import time
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


# ----------------------------
# CONFIG
# ----------------------------
SCOTIA_CARDS = [
    {
        "card_id": "scotia_passport_visa_infinite_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/passport-infinite-card.html"
    },
    {
        "card_id": "scotia_momentum_infinite_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/momentum-infinite-card.html"
    },
    {
        "card_id": "scotia_gold_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/american-express/gold-card.html"
    },
    {
        "card_id": "scotia_scene_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/scene-card.html"
    },
    {
        "card_id": "scotia_momentum_cash_back_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/momentum-cash-back-card.html"
    },
    {
        "card_id": "scotia_momentum_no_fee_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/momentum-no-fee-card.html"
    },
    {
        "card_id": "scotia_no_fee_amex_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/american-express/no-fee-amex-card.html"
    },
    {
        "card_id": "scotia_platinum_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/american-express/platinum-card.html"
    },
    {
        "card_id": "scotia_passport_infinite_privilege_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/passport-infinite-privilege-card.html"
    },
    {
        "card_id": "scotia_momentum_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/mastercard/momentum-card.html"
    },
    {
        "card_id": "scotia_us_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/us-card.html"
    },
    {
        "card_id": "scotia_value_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/value-card.html"
    },
    {
        "card_id": "scotia_scene_student_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/scene-student-card.html"
    },
    {
        "card_id": "scotia_student_value_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/student-value-card.html"
    },
    {
        "card_id": "scotia_student_momentum_cash_back_card_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/student-momentum-cash-back-card.html"
    },
    {
        "card_id": "scotia_student_scotia_momentum_no_fee_visa_ca",
        "url": "https://www.scotiabank.com/ca/en/personal/credit-cards/visa/student-scotia-momentum-no-fee-visa.html"
    }
]

DATA_DIR = Path(r"C:\Users\91951\OneDrive\Desktop\pythonProject\leetcode\Ai-ML-Projects\credit-card-rag\data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "cards_min_scotia.csv"
SNIPPETS_PATH = DATA_DIR / "card_snippets_scotia.jsonl"


# ----------------------------
# BROWSER
# ----------------------------
def get_driver(headless: bool = True):
    chrome_options = Options()
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1800,3200")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=chrome_options
    )
    return driver


# ----------------------------
# HELPERS
# ----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = text.replace("®", "")
    text = text.replace("*", "")
    text = text.replace("™", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def search_first(patterns, text, flags=re.IGNORECASE):
    if not text:
        return ""
    for pattern in patterns:
        m = re.search(pattern, text, flags)
        if m:
            return clean_text(m.group(1) if m.groups() else m.group(0))
    return ""


def normalize_money(value: str) -> str:
    if not value:
        return ""
    return value.replace("$", "").replace(",", "").strip()


def normalize_card_name(name: str) -> str:
    if not name:
        return ""

    name = clean_text(name)
    name = name.replace("Scotiabank®", "Scotiabank")
    name = name.replace("Scene+TM", "Scene+")
    name = name.replace("Scene +", "Scene+")
    name = re.sub(r"\s+", " ", name).strip()

    name = re.sub(r"\s*\|\s*Scotiabank.*$", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*-\s*Scotiabank.*$", "", name, flags=re.IGNORECASE)
    return name.strip()


def extract_section(text: str, start_keywords, end_keywords=None, max_chars=2200):
    if not text:
        return ""

    lower_text = text.lower()
    start_idx = -1

    for kw in start_keywords:
        if not kw:
            continue
        idx = lower_text.find(kw.lower())
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        return ""

    end_idx = len(text)
    if end_keywords:
        for kw in end_keywords:
            if not kw:
                continue
            idx = lower_text.find(kw.lower(), start_idx + 1)
            if idx != -1:
                end_idx = min(end_idx, idx)

    chunk = text[start_idx:end_idx]
    return clean_text(chunk[:max_chars])


def fetch_page_text(driver, url: str, wait_sec: int = 8):
    driver.get(url)
    time.sleep(wait_sec)

    html = driver.page_source
    soup = BeautifulSoup(html, "lxml")
    page_text = clean_text(soup.get_text(" ", strip=True))

    return html, soup, page_text


# ----------------------------
# SCOTIA CARD NAME FIX
# ----------------------------
def scotia_card_name_from_card_id(card_id: str):
    mapping = {
        "scotia_passport_visa_infinite_ca": "Scotiabank Passport Visa Infinite Card",
        "scotia_momentum_infinite_card_ca": "Scotiabank Momentum Visa Infinite Card",
        "scotia_gold_card_ca": "Scotiabank Gold American Express Card",
        "scotia_scene_card_ca": "Scotiabank Scene+ Visa Card",
        "scotia_momentum_cash_back_card_ca": "Scotiabank Momentum Visa Card",
        "scotia_momentum_no_fee_card_ca": "Scotiabank Scotia Momentum No-Fee Visa Card",
        "scotia_no_fee_amex_card_ca": "Scotiabank American Express Card",
        "scotia_platinum_card_ca": "Scotiabank Platinum American Express Card",
        "scotia_passport_infinite_privilege_card_ca": "Scotiabank Passport Visa Infinite Privilege Card",
        "scotia_momentum_card_ca": "Scotiabank Momentum Mastercard Card",
        "scotia_us_card_ca": "Scotiabank U.S. Dollar Visa Card",
        "scotia_value_card_ca": "Scotiabank Value Visa Card",
        "scotia_scene_student_card_ca": "Scotiabank Scene+ Visa Card for Students",
        "scotia_student_value_card_ca": "Scotiabank Value Visa Card for Students",
        "scotia_student_momentum_cash_back_card_ca": "Scotiabank Scotia Momentum Visa Card for Students",
        "scotia_student_scotia_momentum_no_fee_visa_ca": "Scotiabank Scotia Momentum No-Fee Visa Card for Students",
    }
    return mapping.get(card_id, card_id.replace("_", " ").title())


def scotia_card_name_from_url(url: str):
    mapping = {
        "passport-infinite-card": "Scotiabank Passport Visa Infinite Card",
        "momentum-infinite-card": "Scotiabank Momentum Visa Infinite Card",
        "gold-card": "Scotiabank Gold American Express Card",
        "scene-card": "Scotiabank Scene+ Visa Card",
        "momentum-cash-back-card": "Scotiabank Momentum Visa Card",
        "momentum-no-fee-card": "Scotiabank Scotia Momentum No-Fee Visa Card",
        "no-fee-amex-card": "Scotiabank American Express Card",
        "platinum-card": "Scotiabank Platinum American Express Card",
        "passport-infinite-privilege-card": "Scotiabank Passport Visa Infinite Privilege Card",
        "momentum-card": "Scotiabank Momentum Mastercard Card",
        "us-card": "Scotiabank U.S. Dollar Visa Card",
        "value-card": "Scotiabank Value Visa Card",
        "scene-student-card": "Scotiabank Scene+ Visa Card for Students",
        "student-value-card": "Scotiabank Value Visa Card for Students",
        "student-momentum-cash-back-card": "Scotiabank Scotia Momentum Visa Card for Students",
        "student-scotia-momentum-no-fee-visa": "Scotiabank Scotia Momentum No-Fee Visa Card for Students",
    }

    for slug, title in mapping.items():
        if slug in url:
            return title
    return ""


def extract_generic_scotia_card_name(driver, soup, page_text: str, card_cfg: dict):
    card_id = card_cfg["card_id"]
    url = card_cfg["url"]

    mapped_name = scotia_card_name_from_url(url)
    if mapped_name:
        return mapped_name

    if soup.title and soup.title.text:
        title = normalize_card_name(soup.title.text)
        title = title.split("|")[0].strip()

        if (
            "scotiabank" in title.lower()
            and any(x in title.lower() for x in ["visa", "american express", "mastercard", "card", "amex"])
            and len(title) < 180
        ):
            return title

    candidate_selectors = [
        "h1",
        ".page-header h1",
        ".hero-banner__title",
        ".cmp-title__text",
        ".product-name",
        ".card-title",
        ".page-title",
        "[class*='title']",
        "[class*='heading']",
    ]

    seen = set()
    for selector in candidate_selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elems:
                txt = normalize_card_name(elem.text)
                if not txt or txt in seen:
                    continue
                seen.add(txt)

                txt_lower = txt.lower()
                if (
                    "scotiabank" in txt_lower
                    and any(x in txt_lower for x in ["visa", "american express", "mastercard", "card", "amex"])
                    and len(txt) < 180
                ):
                    return txt
        except Exception:
            pass

    patterns = [
        r"(Scotiabank Passport Visa Infinite Privilege Card)",
        r"(Scotiabank Passport Visa Infinite Card)",
        r"(Scotiabank Momentum Visa Infinite Card)",
        r"(Scotiabank Gold American Express Card)",
        r"(Scotiabank Platinum American Express Card)",
        r"(Scotiabank American Express Card)",
        r"(Scotiabank Scene\+ Visa Card for Students)",
        r"(Scotiabank Scene\+ Visa Card)",
        r"(Scotiabank Scotia Momentum No-Fee Visa Card for Students)",
        r"(Scotiabank Scotia Momentum No-Fee Visa Card)",
        r"(Scotiabank Scotia Momentum Visa Card for Students)",
        r"(Scotiabank Momentum Visa Card)",
        r"(Scotiabank Momentum Mastercard Card)",
        r"(Scotiabank U\.S\. Dollar Visa Card)",
        r"(Scotiabank Value Visa Card for Students)",
        r"(Scotiabank Value Visa Card)",
    ]

    for pattern in patterns:
        found = search_first([pattern], page_text)
        if found:
            return normalize_card_name(found)

    return scotia_card_name_from_card_id(card_id)


# ----------------------------
# FIELD EXTRACTION
# ----------------------------
def detect_scotia_network(page_text: str, card_name: str, url: str):
    u = url.lower()
    text = f"{card_name} {page_text}".lower()

    if "/american-express/" in u or "american express" in text or "amex" in text:
        return "American Express"
    if "/mastercard/" in u or "mastercard" in text:
        return "Mastercard"
    if "/visa/" in u or "visa" in text:
        return "Visa"
    return ""


def extract_scotia_annual_fee(page_text: str):
    fee = search_first(
        [
            r"Annual fee\s*\$([0-9]+(?:\.[0-9]{1,2})?)",
            r"Annual Fee[:\s]*\$([0-9]+(?:\.[0-9]{1,2})?)",
        ],
        page_text
    )
    return normalize_money(fee)


def extract_scotia_purchase_rate(page_text: str):
    return search_first(
        [
            r"Purchase interest rate\s*([0-9]+(?:\.[0-9]{1,2})?%)",
            r"Purchase rate\s*([0-9]+(?:\.[0-9]{1,2})?%)",
            r"Interest[:\s]*Purchases[:\s]*([0-9]+(?:\.[0-9]{1,2})?%)",
        ],
        page_text
    )


def extract_scotia_cash_advance_rate(page_text: str):
    return search_first(
        [
            r"Cash advance rate\s*([0-9]+(?:\.[0-9]{1,2})?%)",
            r"Interest[:\s]*Cash Advances[:\s]*([0-9]+(?:\.[0-9]{1,2})?%)",
        ],
        page_text
    )


def extract_scotia_min_credit_limit(page_text: str):
    money = search_first(
        [
            r"Minimum credit limit\s*\$([0-9,]+)",
        ],
        page_text
    )
    return normalize_money(money)


def extract_scotia_supplementary_fee(page_text: str):
    first_card = search_first(
        [
            r"Supplementary cards\s*\$([0-9]+(?:\.[0-9]{1,2})?)\s*/\s*year",
            r"Supplementary cards\s*\$([0-9]+(?:\.[0-9]{1,2})?)",
        ],
        page_text
    )
    return normalize_money(first_card)


def extract_scotia_supplementary_fee_detail(page_text: str):
    return search_first(
        [
            r"(for the first card \$[0-9]+/year for each additional card)",
            r"(\$[0-9]+/year for each additional card)",
            r"(no annual fee on your first supplementary card[^\.;]*)",
        ],
        page_text
    )


def extract_scotia_income_requirement(page_text: str):
    patterns = [
        r"(You have a minimum income of \$[0-9,]+ per year, a minimum household income of \$[0-9,]+, or a minimum of \$[0-9,]+ assets under management\.)",
        r"(minimum income of \$[0-9,]+ per year, a minimum household income of \$[0-9,]+)",
        r"(minimum income of \$[0-9,]+ per year)",
    ]
    return search_first(patterns, page_text)


def extract_scotia_welcome_bonus(page_text: str):
    return search_first(
        [
            r"(Get up to \$[0-9,]+ .*? in value in the first year.*?supplementary card[^\.;]*\.)",
            r"(Get up to \$[0-9,]+ .*? in value in the first year[^\.;]*\.)",
            r"(Get .*?bonus Scene\+ point offer[^\.;]*\.)",
            r"(Earn up to \$[0-9,]+ .*? first year[^\.;]*\.)",
        ],
        page_text
    )


def extract_scotia_rewards_summary(page_text: str):
    patterns = [
        r"(Earn 1 Scene\+ point for every \$1 you spend on eligible travel purchases[^\.;]*)",
        r"(earn an extra 3 Scene\+ points for every \$1 you spend on hotels and car rentals when you book through Scene\+ Travel[^\.;]*)",
        r"(At Sobeys, Safeway, Foodland, participating IGA and co-ops[^\.;]*)",
        r"(On dining out and other eligible grocery stores[^\.;]*)",
        r"(On eligible entertainment purchases[^\.;]*)",
        r"(On eligible transit options[^\.;]*)",
        r"(On all other eligible purchases[^\.;]*)",
        r"(Earn cash back[^\.;]*)",
        r"(Earn points[^\.;]*)",
    ]

    found_parts = []
    for pattern in patterns:
        matches = re.findall(pattern, page_text, re.IGNORECASE)
        for m in matches:
            m = clean_text(m)
            if m and m not in found_parts:
                found_parts.append(m)

    return clean_text(" | ".join(found_parts[:6]))


def extract_scotia_redemption_summary(page_text: str):
    return search_first(
        [
            r"(Apply Scene\+ points to travel.*?redeem your points to cover the full or partial travel cost[^\.;]*\.)",
            r"(Use your Scotiabank Passport Visa Infinite Card to book your vacation[^\.;]*\.)",
            r"(Redeem.*?points[^\.;]*\.)",
            r"(Redeem.*?cash back[^\.;]*\.)",
        ],
        page_text
    )


def infer_best_for(page_text: str, card_name: str, url: str):
    text = f"{card_name} {page_text} {url}".lower()
    tags = []

    if "travel" in text:
        tags.append("travel")
    if "scene+" in text or "scene plus" in text:
        tags.append("scene_plus_rewards")
    if "no foreign transaction fees" in text:
        tags.append("no_fx_fee")
    if "airport lounge" in text:
        tags.append("lounge_access")
    if "dining" in text:
        tags.append("dining")
    if "grocery" in text:
        tags.append("groceries")
    if "entertainment" in text:
        tags.append("entertainment")
    if "transit" in text or "rideshare" in text:
        tags.append("transit")
    if "travel insurance" in text:
        tags.append("travel_insurance")
    if "cash back" in text or "cashback" in text:
        tags.append("cashback")
    if "student" in text:
        tags.append("student")
    if "no-fee" in text or "no fee" in text:
        tags.append("no_annual_fee")
    if "low rate" in text or "value" in text:
        tags.append("low_interest")

    seen = set()
    result = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            result.append(tag)
    return ",".join(result)


def infer_one_liner(page_text: str, card_name: str, url: str):
    text = f"{card_name} {page_text} {url}".lower()

    if "no foreign transaction fees" in text and "airport lounge" in text:
        return "Travel rewards card with no foreign transaction fees, Scene+ points, and airport lounge access."
    if "scene+" in text:
        return "Travel and lifestyle rewards card focused on Scene+ points."
    if "cash back" in text or "cashback" in text:
        return "Cash back credit card for everyday spending."
    if "student" in text:
        return "Student-focused credit card for building credit and everyday spending."
    if "value" in text or "low rate" in text:
        return "Low-interest credit card designed to reduce borrowing costs."
    return "Canadian Scotiabank credit card with rewards and benefits."


# ----------------------------
# MAIN SCRAPER
# ----------------------------
def scrape_scotia_card(driver, card_cfg):
    url = card_cfg["url"]
    card_id = card_cfg["card_id"]

    html, soup, page_text = fetch_page_text(driver, url)

    card_name = extract_generic_scotia_card_name(driver, soup, page_text, card_cfg)
    issuer = "Scotiabank"
    country = "Canada"
    network = detect_scotia_network(page_text, card_name, url)
    card_type = "Credit Card"

    annual_fee = extract_scotia_annual_fee(page_text)
    purchase_rate = extract_scotia_purchase_rate(page_text)
    cash_advance_rate = extract_scotia_cash_advance_rate(page_text)
    min_credit_limit = extract_scotia_min_credit_limit(page_text)
    supplementary_card_fee = extract_scotia_supplementary_fee(page_text)
    supplementary_card_fee_detail = extract_scotia_supplementary_fee_detail(page_text)

    welcome_bonus_summary = extract_scotia_welcome_bonus(page_text)
    eligibility_summary = extract_scotia_income_requirement(page_text)
    rewards_summary = extract_scotia_rewards_summary(page_text)
    redemption_summary = extract_scotia_redemption_summary(page_text)

    best_for = infer_best_for(page_text, card_name, url)
    one_liner = infer_one_liner(page_text, card_name, url)

    overview_text = extract_section(
        page_text,
        start_keywords=[card_name, "Special offer", "Your card at a glance", "At a glance"],
        end_keywords=["Card highlights", "Built-in benefits", "Scene+ points", "Benefits"],
        max_chars=1500
    )

    highlights_text = extract_section(
        page_text,
        start_keywords=["Card highlights", "Highlights"],
        end_keywords=["Built-in benefits", "Scene+ points", "Calculate your rewards", "Benefits"],
        max_chars=1400
    )

    built_in_benefits_text = extract_section(
        page_text,
        start_keywords=["Built-in benefits", "Benefits"],
        end_keywords=["Scene+ points", "Calculate your rewards", "Your award-winning credit card", "Frequently asked questions"],
        max_chars=2200
    )

    rewards_text = extract_section(
        page_text,
        start_keywords=["Scene+ points on every transaction", "Earn Scene+ points for every $1 you spend on eligible purchases", "Earn", "Redeem"],
        end_keywords=["Calculate your rewards", "Your award-winning credit card", "Frequently asked questions"],
        max_chars=2200
    )

    faq_text = extract_section(
        page_text,
        start_keywords=["Frequently asked questions"],
        end_keywords=["Ready to apply?", "Not sure if this is right for you?", "Need more information?"],
        max_chars=1600
    )

    not_right_for_you_text = extract_section(
        page_text,
        start_keywords=["Not sure if this is right for you?", "This card might not be right if you:"],
        end_keywords=["Explore other credit cards", "Need more information?"],
        max_chars=1400
    )

    card_setup_text = extract_section(
        page_text,
        start_keywords=["Need more information?", "Set up your card", "Get to know your card"],
        end_keywords=["Scene+ program Terms", "Legal", "Conditions"],
        max_chars=1600
    )

    fees_text = clean_text(
        f"Annual fee: ${annual_fee}; "
        f"Minimum credit limit: ${min_credit_limit}; "
        f"Purchase interest: {purchase_rate}; "
        f"Cash advance interest: {cash_advance_rate}; "
        f"Supplementary card fee: ${supplementary_card_fee}. "
        f"{supplementary_card_fee_detail}"
    )

    card_record = {
        "card_id": card_id,
        "card_name": card_name,
        "issuer": issuer,
        "country": country,
        "network": network,
        "card_type": card_type,
        "link": url,
        "monthly_fee": "",
        "annual_fee": annual_fee,
        "welcome_bonus_summary": welcome_bonus_summary,
        "best_for": best_for,
        "one_liner": one_liner,
        "eligibility_summary": eligibility_summary,
        "purchase_rate": purchase_rate,
        "cash_advance_rate": cash_advance_rate,
        "minimum_credit_limit": min_credit_limit,
        "supplementary_card_fee": supplementary_card_fee,
        "supplementary_card_fee_detail": supplementary_card_fee_detail,
        "rewards_summary": rewards_summary,
        "redemption_summary": redemption_summary,
    }

    snippets = [
        {"chunk_id": f"{card_id}_overview", "card_id": card_id, "section": "overview", "text": overview_text},
        {"chunk_id": f"{card_id}_highlights", "card_id": card_id, "section": "highlights", "text": highlights_text},
        {"chunk_id": f"{card_id}_built_in_benefits", "card_id": card_id, "section": "built_in_benefits", "text": built_in_benefits_text},
        {"chunk_id": f"{card_id}_rewards", "card_id": card_id, "section": "rewards", "text": rewards_text},
        {"chunk_id": f"{card_id}_welcome_bonus", "card_id": card_id, "section": "welcome_bonus", "text": welcome_bonus_summary},
        {"chunk_id": f"{card_id}_faq", "card_id": card_id, "section": "faq", "text": faq_text},
        {"chunk_id": f"{card_id}_not_right_for_you", "card_id": card_id, "section": "not_right_for_you", "text": not_right_for_you_text},
        {"chunk_id": f"{card_id}_card_setup", "card_id": card_id, "section": "card_setup", "text": card_setup_text},
        {"chunk_id": f"{card_id}_fees", "card_id": card_id, "section": "fees", "text": fees_text},
        {"chunk_id": f"{card_id}_eligibility", "card_id": card_id, "section": "eligibility", "text": eligibility_summary},
    ]

    snippets = [s for s in snippets if s["text"]]

    return card_record, snippets


# ----------------------------
# RUN
# ----------------------------
def main():
    driver = get_driver(headless=True)

    all_cards = []
    all_snippets = []

    try:
        for card in SCOTIA_CARDS:
            print(f"Scraping: {card['card_id']}")
            card_record, snippets = scrape_scotia_card(driver, card)

            print(f"{card['card_id']} -> {card_record['card_name']}")
            print(card["url"])
            print("-" * 80)

            all_cards.append(card_record)
            all_snippets.extend(snippets)
            time.sleep(2)
    finally:
        driver.quit()

    cards_df = pd.DataFrame(all_cards)
    cards_df.to_csv(CSV_PATH, index=False)

    with open(SNIPPETS_PATH, "w", encoding="utf-8") as f:
        for item in all_snippets:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print("Saved CSV to:", CSV_PATH)
    print("Saved snippets to:", SNIPPETS_PATH)
    print(cards_df[["card_id", "card_name", "annual_fee", "network"]].T)

    print("\nSnippet preview:")
    for s in all_snippets[:20]:
        print(f"- {s['section']}: {s['text'][:220]}...")


if __name__ == "__main__":
    main()

Scraping: scotia_passport_visa_infinite_ca
scotia_passport_visa_infinite_ca -> Scotiabank Passport Visa Infinite Card
https://www.scotiabank.com/ca/en/personal/credit-cards/visa/passport-infinite-card.html
--------------------------------------------------------------------------------
Scraping: scotia_momentum_infinite_card_ca
scotia_momentum_infinite_card_ca -> Scotiabank Momentum Visa Infinite Card
https://www.scotiabank.com/ca/en/personal/credit-cards/visa/momentum-infinite-card.html
--------------------------------------------------------------------------------
Scraping: scotia_gold_card_ca
scotia_gold_card_ca -> Scotiabank Gold American Express Card
https://www.scotiabank.com/ca/en/personal/credit-cards/american-express/gold-card.html
--------------------------------------------------------------------------------
Scraping: scotia_scene_card_ca
scotia_scene_card_ca -> Scotiabank Scene+ Visa Card
https://www.scotiabank.com/ca/en/personal/credit-cards/visa/scene-card.html
--------